# 03 · Evaluate (baselines and LoRA) on Colab

Runs `scripts/eval.py` with the fixed prompts in `eval/prompts.json`.

1. Run the **baselines** (conditions A and B) once, ideally *before* training.
2. After picking a checkpoint, run the **full eval** (A, B, C at 0.4/0.7/1.0, D).

Results (images, grids, `scores.csv`, `summary.csv`) are written to Drive.
The full eval is 180 images: roughly 30-40 minutes on an L4, about 1 compute unit.

In [ ]:
# --- Check the GPU before spending units ---------------------------------
# Runtime -> Change runtime type -> L4. An A100 works but burns units several times faster.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Mount Drive and open the project (lives in MyDrive/patent-lora/repo) ----
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, zipfile
DRIVE_ROOT = "/content/drive/MyDrive/patent-lora"   # project, checkpoints and results all live here
REPO_DIR   = f"{DRIVE_ROOT}/repo"
ZIP_PATH   = f"{DRIVE_ROOT}/patent-drawing-lora.zip"  # upload the project zip here once
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError("Upload patent-drawing-lora.zip to MyDrive/patent-lora/ in Google Drive, then re-run.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DRIVE_ROOT)
    shutil.move(f"{DRIVE_ROOT}/patent-drawing-lora", REPO_DIR)
    print("Project unpacked to", REPO_DIR)
%cd {REPO_DIR}

In [ ]:
DIFFUSERS_COMMIT = "cda991a69ad21b8d3239a307d58538fa1a3a6b00"
!pip -q install "git+https://github.com/huggingface/diffusers@{DIFFUSERS_COMMIT}"
!pip -q install "transformers>=4.25.1" "peft>=0.17.0" accelerate open_clip_torch
RESULTS = f"{DRIVE_ROOT}/eval"

## Baselines (no LoRA)

In [ ]:
!python scripts/eval.py --run baselines --baselines-only --clip --out {RESULTS}

## Full eval with your chosen checkpoint
Set `CKPT` to the checkpoint folder you picked in notebook 02.

In [ ]:
CKPT = f"{DRIVE_ROOT}/output/run01_r16_lr1e-4/checkpoint-1000"   # <- change to your pick
RUN  = "lora_" + CKPT.rstrip("/").rsplit("/", 1)[-1]
!python scripts/eval.py --run {RUN} --lora {CKPT} --clip --out {RESULTS}

In [ ]:
import pandas as pd
pd.read_csv(f"{RESULTS}/{RUN}/summary.csv")

## Publish the LoRA to the Hugging Face Hub (optional)
Needs a Hugging Face **write** token: add it in Colab's Secrets panel (key icon) as `HF_TOKEN`.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login
login(token=userdata.get("HF_TOKEN"))
HUB_REPO = "sisodiajatin/patent-drawing-style-lora"   # <- your Hub username / repo name
api = HfApi()
api.create_repo(HUB_REPO, exist_ok=True)
api.upload_file(path_or_fileobj=f"{CKPT}/pytorch_lora_weights.safetensors",
                path_in_repo="pytorch_lora_weights.safetensors", repo_id=HUB_REPO)
api.upload_file(path_or_fileobj="demo/MODEL_CARD.md", path_in_repo="README.md", repo_id=HUB_REPO)
print("https://huggingface.co/" + HUB_REPO)

In [ ]:
from google.colab import runtime
runtime.unassign()   # disconnect when done